In [1]:
# ============================================
# Burger Sales Forecasting Project
# Step 1: Import Required Libraries
# ============================================

# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt

# Display plots inside the notebook
%matplotlib inline

# Display all columns without truncation
pd.set_option("display.max_columns", None)

# Display wider tables
pd.set_option("display.width", 1000)

# Ignore unnecessary warning messages
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# ============================================
# Load Dataset
# ============================================

# Read the CSV file
df = pd.read_csv(r"C:\Users\pc\Desktop\Burger_Sales_Forecasting\data\burger_data.csv")

# Display the first five rows
df.head()

,Date,Region,Temperature,Humidity,Wind,Visibility,Pressure,Sales
0,15-09-2020,Reg1,10.248814,0.779164,11.509130,14.503403,1017.293917,991.60
1,14-09-2020,Reg1,10.337595,0.908549,7.432656,2.232960,1019.452636,1858.59
2,13-09-2020,Reg1,20.763686,0.505324,7.788249,4.779211,1022.677119,3.99
3,12-09-2020,Reg1,21.500892,0.758557,3.767432,9.904534,1009.341357,3090.78
4,11-09-2020,Reg1,21.774269,0.398296,20.705369,15.224605,1015.713234,990.99


In [3]:
# ============================================
# Dataset Dimensions
# ============================================

# Display the number of rows and columns
print("Dataset Shape:", df.shape)

Dataset Shape: (24500, 8)


In [4]:
# ============================================
# Display Column Names
# ============================================

print("Columns in the dataset:\n")

for column in df.columns:
    print(column)

Columns in the dataset:

Date
Region
Temperature
Humidity
Wind
Visibility
Pressure
Sales


In [5]:
# ============================================
# Dataset Information
# ============================================

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24500 entries, 0 to 24499
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         24500 non-null  object 
 1   Region       24500 non-null  object 
 2   Temperature  24500 non-null  float64
 3   Humidity     24500 non-null  float64
 4   Wind         24500 non-null  float64
 5   Visibility   24500 non-null  float64
 6   Pressure     24500 non-null  float64
 7   Sales        24500 non-null  float64
dtypes: float64(6), object(2)
memory usage: 1.5+ MB


In [6]:
# ============================================
# Missing Values
# ============================================

missing_values = df.isnull().sum()

print("Missing Values Per Column:\n")
print(missing_values)

Missing Values Per Column:

Date           0
Region         0
Temperature    0
Humidity       0
Wind           0
Visibility     0
Pressure       0
Sales          0
dtype: int64


In [7]:
# ============================================
# Summary Statistics
# ============================================

df.describe()

,Temperature,Humidity,Wind,Visibility,Pressure,Sales
count,24500.000000,24500.000000,24500.000000,24500.000000,24500.000000,24500.000000
mean,13.082508,0.718518,10.697779,9.997047,1004.846838,1872.654344
std,9.564111,0.199462,6.745872,4.179089,91.983542,1085.555366
min,-7.170076,0.000000,0.000000,0.000000,0.000000,0.160000
25%,4.353792,0.547797,5.423364,6.861189,1012.521059,846.725000
50%,13.992684,0.734187,9.403089,10.553807,1016.473950,2022.510000
75%,21.803201,0.902328,14.985763,13.605723,1020.807589,2894.247500
max,33.538482,0.999396,36.206303,16.093742,1032.238678,3564.540000


In [8]:
# ============================================
# Convert Date Column to Datetime
# ============================================

# Convert the 'Date' column from text to datetime format.
# The format in the dataset is day-month-year (e.g., 15-09-2020).
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")

# Verify the conversion by displaying the data types.
print(df.dtypes)

Date           datetime64[ns]
Region                 object
Temperature           float64
Humidity              float64
Wind                  float64
Visibility            float64
Pressure              float64
Sales                 float64
dtype: object


In [9]:
# ============================================
# Sort Dataset by Region and Date
# ============================================

# Sort the dataset by Region first, then by Date in ascending order.
# Time series models require chronological order.
df = df.sort_values(by=["Region", "Date"])

# Reset the index after sorting.
df.reset_index(drop=True, inplace=True)

# Display the first five rows after sorting.
df.head()

,Date,Region,Temperature,Humidity,Wind,Visibility,Pressure,Sales
0,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
1,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
2,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
3,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
4,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45


In [10]:
# This confirms the data spans the period described in the project documentation.
# # ============================================
# Check Date Range
# ============================================

print("Earliest Date :", df["Date"].min())
print("Latest Date   :", df["Date"].max())

Earliest Date : 2014-01-01 00:00:00
Latest Date   : 2020-09-15 00:00:00


The documentation mentions there are 10 regions. We want to verify that:

All regions are present.
Each region has roughly the same number of observations.

In [11]:
# ============================================
# Records Available per Region
# ============================================

print("Number of records in each region:\n")

print(df["Region"].value_counts().sort_index())

Number of records in each region:

Region
Reg1     2493
Reg10    2442
Reg2     2443
Reg3     2446
Reg4     2446
Reg5     2444
Reg6     2448
Reg7     2447
Reg8     2445
Reg9     2446
Name: count, dtype: int64


The first five rows are completely identical.

## That immediately raises a question:

Are these true duplicate records?
Or are there multiple stores within Region 1 that happen to share the same weather and sales?
Or did something happen during sorting?

We must investigate before moving on, because duplicate data can bias our analysis and model.

In [12]:
# ============================================
# Check for Duplicate Rows
# ============================================

# Count duplicate rows
duplicate_rows = df.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate rows: 48


In [13]:
# ============================================
# Display Duplicate Rows
# ============================================

duplicates = df[df.duplicated()]

duplicates.head(10)

,Date,Region,Temperature,Humidity,Wind,Visibility,Pressure,Sales
1,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
2,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
3,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
4,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
5,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
6,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
8,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
9,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
10,2014-01-01,Reg1,17.471199,0.753878,6.755839,13.807525,1014.437661,1539.45
15,2014-01-05,Reg1,6.366205,0.859343,8.104632,7.563892,1018.752793,1139.66


In [14]:
# ============================================
# Records Per Date for Region 1
# ============================================

df[df["Region"] == "Reg1"]["Date"].value_counts().sort_index().head(10)

Date
2014-01-01    11
2014-01-02     1
2014-01-03     1
2014-01-04     1
2014-01-05    11
2014-01-06     1
2014-01-07     1
2014-01-08     1
2014-01-09     1
2014-01-10     1
Name: count, dtype: int64

i found:

48 duplicate rows
Some dates (e.g 2014-01-01) appear 11 times in Reg1.
Most other dates appear only once.

This indicates the duplicates are not random—they're concentrated on a few specific dates.

## Data Cleaning

Before building predictive models, the dataset was inspected for duplicate records.
A total of **48 duplicate rows** were identified. Since these records were identical across all columns, they were removed to avoid introducing bias into the forecasting models.
Before deleting anything, it's good to keep a record of how many rows were removed.

In [15]:
# ============================================
# Remove Duplicate Records
# ============================================

# Store the original number of rows
original_rows = len(df)

# Remove duplicate rows
df = df.drop_duplicates()

# Reset the index after removing duplicates
df.reset_index(drop=True, inplace=True)

# Store the new number of rows
new_rows = len(df)

# Display the results
print(f"Original number of rows : {original_rows}")
print(f"Rows after cleaning     : {new_rows}")
print(f"Duplicates removed      : {original_rows - new_rows}")

Original number of rows : 24500
Rows after cleaning     : 24452
Duplicates removed      : 48


In [16]:
# Verify duplicate removal
print("Remaining duplicate rows:", df.duplicated().sum())

Remaining duplicate rows: 0


In [17]:
# =====================================================
# Save the Clean Dataset
# =====================================================

from pathlib import Path

# Ensure the target directory exists
output_dir = Path("../data")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset for use in later notebooks
output_path = output_dir / "burger_data_clean.csv"
df.to_csv(output_path, index=False)

print(f"Clean dataset saved successfully at {output_path}")

Clean dataset saved successfully at ..\data\burger_data_clean.csv
